In [7]:
!pip install transformers sentencepiece accelerate torch gradio --quiet


In [8]:
import re
import torch
from typing import List, Tuple
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

MODEL = "t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(DEVICE)
model.eval()

print("Model Loaded on:", DEVICE)


Model Loaded on: cuda


In [9]:
CODE_FENCE_RE = re.compile(r"```[\s\S]*?```", re.MULTILINE)
INLINE_CODE_RE = re.compile(r"`[^`]+`")
MATH_BLOCK_RE = re.compile(r"\$\$[\s\S]*?\$\$")
MATH_INLINE_RE = re.compile(r"\$(.+?)\$")

def extract_placeholders(text: str) -> Tuple[str, List[Tuple[str, str]]]:
    replacements = []
    counter = 0

    def _replace(match):
        nonlocal counter
        token = f"[[P{counter}]]"
        replacements.append((token, match.group(0)))
        counter += 1
        return token

    t = CODE_FENCE_RE.sub(_replace, text)
    t = MATH_BLOCK_RE.sub(_replace, t)
    t = MATH_INLINE_RE.sub(_replace, t)
    t = INLINE_CODE_RE.sub(_replace, t)

    return t, replacements

def restore_placeholders(text: str, replacements: List[Tuple[str, str]]):
    for token, original in replacements:
        text = text.replace(token, original)
    return text


In [10]:
def build_prompt(masked_text: str):
    # Correct minimal prompt for T5 — REQUIRED
    return "simplify: " + masked_text


In [11]:
GEN_ARGS = {
    "num_beams": 4,
    "max_length": 256,
    "no_repeat_ngram_size": 3,
    "early_stopping": True
}

def simplify_text(text: str):
    if not text or text.strip() == "":
        return ""

    # Step 1: Mask code + math
    masked, repl = extract_placeholders(text)

    # Step 2: Build prompt (T5-friendly)
    prompt = build_prompt(masked)

    # Step 3: Generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(**inputs, **GEN_ARGS)

    simplified = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 4: Remove possible prompt echo
    if simplified.startswith("simplify:"):
        simplified = simplified.replace("simplify:", "", 1).strip()

    # Step 5: Restore preserved content
    final = restore_placeholders(simplified, repl)

    return final.strip()


In [12]:
def simplify_interface(text):
    return simplify_text(text)

demo = gr.Interface(
    fn=simplify_interface,
    inputs=gr.Textbox(lines=10, label="Enter CSE text (algorithms, code, formulas)"),
    outputs=gr.Textbox(lines=10, label="Simplified Explanation"),
    title="CSE Text Simplifier (T5-Base – Encoder–Decoder)",
    description="Simplifies technical explanations while preserving code blocks and mathematical formulas."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc8b795b83c827e2cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
